# Explaining Nearest-Neighbor Classifiers: Introduction

When explaining nearest-neighbor classifiers, the goal is, given a model trained on some training dataset $D$, to quantify how much each training data point $z_i = (x_i, y_i) \in D$ contributes to the model predicting the 'explanation class' $y_\text{explain}$ on some explanation point $x_\text{explain}$. For this purpose we can define, in the case of a simple $k$-nearest neighbors classifier, the utility of a coalition $S \subseteq D$ as the likelihood of predicting the class $y_\text{explain}$ [\[Jia19\]](../citations.rst):
$$
    \nu(S) = \frac{1}{k} \sum_{j=1}^{\min\{k, |S|\}} \chi(y_{\alpha_{S,j}} = y_\text{explain}),
$$
where $\alpha_{S,j}$ is the index of the $j$-nearest training point to $x_\text{explain}$ in the coalition $S$, and $\chi(P)$ is the indicator function, defined as
$$
    \chi(P) = \left\lbrace
        \begin{array}{ll}
            1 & \text{if } P \\
            0 & \text{otherwise.} \\
        \end{array}
    \right.
$$

While `shapiq` already offers a way to explain nearest-neighbor models using `ExactComputer`, it requires evaluating the utility function for all possible coalitions, resulting in an exponential runtime. However, special properties of nearest-neighbor models can be exploited for designing more efficient algorithms, which are implemented in this library.

Here is an overview of the three kinds of nearest-neighbor classifiers for which `shapiq_student` implements explainers:

| Classifier | `sklearn` implementation | Basis for explanation algorithm |
|---|---|---|
| Unweighted $k$-nearest neighbor classifier | `KNeighborsClassifier(weights="uniform")`, see [here](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) | [\[Jia19\]](../citations.rst) |
| Weighted $k$-nearest neighbor classifier  | `KNeighborsClassifier(weights="distance")`, see [here](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) | [\[Wng24\]](../citations.rst)|
| Threshold nearest neighbor classifier | `RadiusNeighborsClassifier()`, see [here](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.RadiusNeighborsClassifier.html) | [\[Wng23\]](../citations.rst) |

For this notebook, we will limit ourselves to the simplest option, the unweighted $k$-nearest neighbors (KNN) classifier. The other two kinds of classifiers are discussed in the [follow-up notebook](./explainers_2.ipynb).

Let's start with some setup. First, we generate a synthetic classification dataset with some non-linearity, making it suitable for nearest-neighbor classification. The dataset has two classes and just two features to simplify the presentation. We will plot the dataset using the function `plot_datasets`, which we defined in a separate file since we don't want to bore you with the details.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

from plot import plot_datasets

X_train, y_train = make_classification(
    n_samples=30,
    n_features=2,
    n_redundant=0,
    n_clusters_per_class=1,
    n_informative=2,
    n_classes=2,
    random_state=45,
)

fig, ax = plt.subplots(figsize=(6, 6))
plot_datasets(ax, X_train, y_train)
print(f"Size of training dataset: {X_train.shape[0]}")

Now, let's fit a KNN model to the training data. Then, we can define an explanation data point $x_\text{explain}$ and let the model predict its class.

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)

x_explain = np.array([[-0.75, -0.4]])
y_explain_pred = model.predict(x_explain)[0]
print(y_explain_pred)

y_explain_proba = model.predict_proba(x_explain)
print(y_explain_proba)

So our model predicted the class with index `0` with a probability of around 66%. But why? &mdash; or, more specifically, which training data points caused it to make this prediction? This is where explainers come into play.

## Explaining a Single Data Point

Let's create an explainer for the model we just defined. The `KNNExplainer` class is the main interface to `shapiq_student`'s explainers. It automatically selects the right explainer subclass depending on the class of model provided. In this case, we are passing an unweighted KNN model, so `NormalKNNExplainer` will be selected. When instantiating an explainer, we also need to provide `class_index`, i.e. the index of the class to explain. This corresponds to the variable $y_\text{explain}$ mentioned earlier.

In [ ]:
from shapiq_student import KNNExplainer

explainer = KNNExplainer(model, class_index=y_explain_pred)
explainer.__class__

Note that we set `class_index=y_test_pred`, since for now, we want to quantify the contribution of the training data to the class that was actually predicted. (We could also set a different class index if we wished to see how much the data points contribute to shifting the prediction towards another class.)

In order to obtain an explanation, we simply call the `explain()` method of the explainer with the test datapoint. This will return an `InteractionValues` object containing the Shapley Values for each datapoint in the training data set.

In [ ]:
iv = explainer.explain(x_explain)
print(iv)

The `explain()` method implements the algorithm proposed in [\[Jia19\]](../citations.rst) for calculating Shapley Values for unweighted $k$-nearest neighbor models in **log-linear time**. To do so, it inspects the training data of the `sklearn` model that was passed in the constructor, sorts the training data points by their distance to $x_\text{test}$ and finally computes Shapley Values in linear time.

For easier handling we can convert the `InteractionValues` object to a simple numpy array, since we only have interactions of order 1. To do so, we use a utility function provided by `shapiq_student`,

In [ ]:
from shapiq_student.explainer.knn import interaction_values_to_array

sv = interaction_values_to_array(iv)
print(sv)

If we sum up all Shapley Values, we should get our original model prediction for class index `0`, which was 66%:

In [ ]:
print(np.sum(sv))

To get a clear view of the result we can visualize the top 10 absolute values in a bar plot:

In [ ]:
top_n = 10
top_n_ivs, _ = iv.get_top_k(top_n, as_interaction_values=False)
# Sort by absolute value, descending
top_n_ivs_sorted = sorted(top_n_ivs.items(), key=lambda t: -np.abs(t[1]))
top_n_idxs = np.array([idx for (idx,), _ in top_n_ivs_sorted], dtype=int)
top_n_svs = np.array([sv for _, sv in top_n_ivs_sorted], dtype=float)

width = 0.5
fig, ax = plt.subplots(figsize=(6, 6))
for class_ in set(y_train):
    xs = np.where(y_train[top_n_idxs] == class_)[0]
    ys = top_n_svs[xs]
    ax.bar(xs, ys, width=width, label=f"Class {class_}")

ax.set_xticks(np.arange(top_n), top_n_idxs)

ax.axhline(y=0, linewidth=1, color="black")
ax.set_xlabel("Training Data Point")
ax.set_ylabel("Shapley Value")
ax.legend()
ax.title.set_text(f"Top {top_n} Shapley Values by Magnitude")

This is still not very helpful, so let's visualize the Shapley Values by plotting the training data set again, this time setting the size of each dot according to the Shapley Value attributed to the training data point it represents. The `shapiq_student` library provides the function `plot_points_shapley_2d` for just this purpose.

In [ ]:
import matplotlib.pyplot as plt

from shapiq_student.plot import plot_points_shapley_2d

fig, ax = plt.subplots(figsize=(6, 6))
plot_points_shapley_2d(ax, X_train, y_train, sv, set(y_train), x_explain, scale=1)

The plot represents each training data point as a circle colored according to its class and scaled by its absolute Shapley Value. Positive values are shown as filled circles and negative values as empty circles.

As we can observe, training data points whose class agrees with `class_index` have positive Shapley Values while the rest have negative Shapley Values.
Furthermore, points that are closer to the explanation point $x_\text{explain}$ have higher absolute Shapley Values than the ones farther away. This makes sense, since the points closest to $x_\text{explain}$ have the highest chance of being among the $k$-nearest points, thereby influencing the prediction of the model, while points farther away can rarely change the prediction.

## Data Valuation for KNN

In the context of data valuation, we want to quantify the usefulness of training data to a model's performance. Shapley Values provide a simple way to achieve this goal. To estimate the usefulness of each point of a training data set, we calculate Shapley Values for a set of test data points and average the results.

First, let's create a classification data set and split it into train and test sets. We will corrupt the training data by changing the class of a few randomly selected data points. Later, we will observe how we can identify corrupted or useless training data with Shapley Values.

In [ ]:
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=100,
    n_features=2,
    n_redundant=0,
    n_clusters_per_class=1,
    n_informative=2,
    n_classes=2,
    flip_y=0,
    random_state=49,
    class_sep=1.5,
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

y_train_corrupted = y_train.copy()
n_corrupt = 7
rng = np.random.default_rng(seed=43)
corrupted = rng.choice(np.arange(X_train.shape[0]), size=n_corrupt, replace=False)
# Since our only class indices are 0 and 1, this is a quick way to change the class
y_train_corrupted[corrupted] = 1 - y_train[corrupted]

fig, ax = plt.subplots(figsize=(6, 6))
plot_datasets(ax, X_train, y_train_corrupted, X_test, y_test)
# Mark corrupted datapoints
ax.scatter(
    X_train[corrupted, 0],
    X_train[corrupted, 1],
    marker="o",
    edgecolors="#b1170c",
    facecolors="none",
    s=100,
);

Now, let's compute Shapley Values based on the entire test dataset by averaging the Shapley Values computed using each test point.

In [ ]:
# Train the model with the corrupted training data
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train_corrupted)
explainer = KNNExplainer(model, class_index=0)

sv_test = np.zeros(X_train.shape[0], dtype=np.float64)

for x_test_current, y_test_current in zip(X_test, y_test, strict=True):
    iv = explainer.explain(x_test_current, class_index=y_test_current)
    sv_current = interaction_values_to_array(iv)
    sv_test += sv_current

sv_test /= X_test.shape[0]

We can reasonably assume that the corrupted training data points will on average make the model's prediction worse, resulting in negative Shapley Values, so let's filter out just those indices where the Shapley Value is below zero and compare with our original array of corrupted indices:

In [ ]:
print(f"Corrupted: {np.sort(corrupted)}")  # sort for easier comparison
print(f"Negative Shapley Values: {np.where(sv_test < 0)[0]}")

Et voilà! We have successfully identified the corrupted samples -- with one exception being the data point at index 20, call it $x_{20}$. Let's plot the data set again and mark $x_{20}$.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_datasets(ax, X_train, y_train_corrupted, X_test, y_test)
# Mark one specific point
i = 20
ax.scatter(X_train[i, 0], X_train[i, 1], marker="o", edgecolors="red", facecolors="none", s=100);

If we look closely at how the test dataset is distributed, we can observe that no test point is close to $x_{20}$, and more importantly there are a lot of 'correct' training data points between $x_{20}$ and the closest test data points. This means that it rarely got to influence the model prediction at all, which may explain why its Shapley Value didn't get shifted much toward the negative.

## Up next

In this notebook, we saw how the `KNNExplainer` class can explain model predictions for $k$-nearest neighbor models and how it can be used for data valuation. To find out how to explain weighted $k$-nearest neighbor and threshold nearest neighbor models, continue with the [next notebook](./explainers_2.ipynb).